In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.io import loadmat

from cqpsolver import Problem, Solver, SolverState

%load_ext snakeviz

In [ ]:
mat_dict: dict[np.ndarray] = loadmat("../QP-Test-Problems/MAT_Files/EXDATA.mat")

Q: sp.csc_array = sp.csc_array(mat_dict["Q"].astype(float))
q: np.ndarray = mat_dict["c"].astype(float)
A: sp.csc_array = sp.csc_array(mat_dict["A"].astype(float))
rl: np.ndarray = mat_dict["rl"].astype(float).flatten()
ru: np.ndarray = mat_dict["ru"].astype(float).flatten()
lb: np.ndarray = mat_dict["lb"].astype(float).flatten().reshape(-1, 1)
ub: np.ndarray = mat_dict["ub"].astype(float).flatten().reshape(-1, 1)

In [3]:
eq_mask: np.ndarray = rl == ru
A_eq: sp.csc_array = sp.csc_array(A[eq_mask])
b_eq: np.ndarray = ru[eq_mask].reshape(-1, 1)

A_eq: sp.csc_array = A_eq if A_eq.size > 0 else sp.csc_array((0, A.shape[1]))
b_eq: np.ndarray = b_eq if b_eq.size > 0 else np.zeros((0, 1))

ineq_mask: np.ndarray = np.invert(eq_mask)
G_ineq: sp.csc_array = sp.vstack([A[ineq_mask], -A[ineq_mask]], format="csc")
h_ineq: np.ndarray = np.concatenate([ru[ineq_mask], -rl[ineq_mask]]).reshape(-1, 1)

n: int = Q.shape[0]
G_full: sp.csc_array = sp.vstack([G_ineq, sp.eye(n), -sp.eye(n)], format="csc")
h_full: np.ndarray = np.vstack([h_ineq, ub, -lb])

finite_mask: np.ndarray = np.isfinite(h_full).flatten()
G: sp.csc_array = sp.csc_array(G_full[finite_mask])
h: np.ndarray = (h_full[finite_mask]).reshape(-1, 1)

In [4]:
prob: Problem = Problem(Q, q, G, h, A_eq, b_eq)
solver: Solver = Solver(prob, max_iter=100, quiet=False)
state_history: list[SolverState] = solver.solve()
# %snakeviz -t state_history: list[SolverState] = solver.solve()

─────────────────────────────────────────────────────────────────────────────────────────────────────
Iter. │   Objective    │ Primal Inequality │ Primal Equality │ Stationarity │   Duality   │ Step Size
─────────────────────────────────────────────────────────────────────────────────────────────────────
  0   │   -28.366678   │    1.3490e+00     │   5.2736e-15    │  3.3947e+01  │ 1.1197e+05  │     —    
  1   │   160.28426    │    4.7011e-02     │   5.2180e-15    │  1.1830e+00  │ 4.4434e+03  │  0.9652  
  2   │   118.93473    │    1.3639e-02     │   1.1574e-14    │  3.4321e-01  │ 1.5572e+03  │  0.7099  
  3   │   85.554805    │    7.7579e-03     │   2.2982e-14    │  1.9523e-01  │ 1.1110e+03  │  0.4312  
  4   │   27.822853    │    3.8915e-03     │   2.6090e-14    │  9.7929e-02  │ 6.5784e+02  │  0.4984  
  5   │   -2.938458    │    2.7712e-03     │   6.8723e-14    │  6.9737e-02  │ 5.1189e+02  │  0.2879  
  6   │   -28.450247   │    1.7207e-03     │   2.5147e-14    │  4.3301e-02  │ 3.88